In [1]:
%pip install mediapipe==0.10.14
%pip install protobuf==4.25.3
%pip install numpy==1.26.4
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/413.4 kB ? eta -:--:--
      --------------------------------------- 10.2/413.4 kB ? eta -:--:--
     -- ---------------------------------- 30.7/413.4 kB 325.1 kB/s eta 0:00:02
     -- ---------------------------------- 30.7/413.4 kB 325.1 kB/s eta 0:00:02
     --- --------------------------------- 41.0/413.4 kB 217.9 kB/s eta 0:00:02
     ----- ------------------------------- 61.4/413.4 kB 252.2 kB/s eta 0:00:02
     -------- ---------------------------- 92.2/413.4 kB 308.0 kB/s eta 0:00:02
     --------- -------------------------- 112.6/413.4 kB 385.0 kB/s eta 0:00:01
     ---------- ------------------------- 122.9/413.4 kB 343.4 kB/s eta 0:00:01
     ---------------- ------------------- 194.6/413.4 kB 454.0 kB/s eta 0:00:01
     ------------------------ ----------- 276.5/413.4 kB 607.9 kB/s eta 0:00:01
     ------------------------ ----------- 286.7/413.4 kB 553.0 kB/s eta 0:00:01
     -------------------------------- --- 378.9/413.4


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     --------------------------------------- 0.0/15.8 MB 330.3 kB/s eta 0:00:48
     --------------------------------------- 0.0/15.8 MB 330.3 kB/s eta 0:00:48
     --------------------------------------- 0.0/15.8 MB 178.6 kB/s eta 0:01:29
     --------------------------------------- 0.1/15.8 MB 281.8 kB/s eta 0:00:56
     --------------------------------------- 0.1/15.8 MB 368.6 kB/s eta 0:00:43
     --------------------------------------- 0.1/15.8 MB 343.4 kB/s eta 0:00:46
      -------------------------------------- 0.2/15.8 MB 498.9 kB/s eta 0:00:32
      -------------------------------------- 0.3/15.8 MB 585.8 kB/s eta 0:00:27
      -------------------------------------- 0.3/15.8 MB 585.8 kB/s eta 0:00:27
      -------------------------------------- 0.4/15.8 MB 675.0 kB/s eta 0:00:23
     - ------------------------------------- 0.5/15.8 MB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
"""
extract_to_test.py
──────────────────
Scans any folder of videos (flat or class-subfolders) and extracts
MediaPipe Holistic landmarks into .npy files saved under  test/

Usage:
    python extract_to_test.py <path_to_video_folder>

Examples:
    python extract_to_test.py data/JINA
    python extract_to_test.py C:/my_videos
    python extract_to_test.py data            ← scans all word sub-folders

Output structure:
    test/
     └── <word_or_flat>/
           └── <video_stem>.npy   shape: (120, 141)
"""

import sys
import os
import cv2
import numpy as np
import mediapipe as mp
from pathlib import Path

# ── Config (matches data.ipynb exactly) ─────────────────────────────────────
MAX_SEQ_LENGTH = 120
FEATURES       = 141
POSE_IDX       = [0, 11, 12, 13, 14]   # nose, l-shoulder, r-shoulder, l-elbow, r-elbow
VIDEO_EXT      = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
OUTPUT_ROOT    = Path("test")

mp_holistic = mp.solutions.holistic


# ── Helper functions (identical to data.ipynb) ───────────────────────────────
def normalize_frame(vec):
    vec  = vec.copy()
    pose = vec[:15].reshape(5, 3)
    lh   = vec[15:78].reshape(21, 3)
    rh   = vec[78:].reshape(21, 3)

    if np.any(pose):
        ls     = pose[1]
        rs     = pose[2]
        center = (ls + rs) / 2
        dist   = np.linalg.norm(ls - rs)
        if dist > 1e-6:
            pose = (pose - center) / dist
            if np.any(lh):
                lh = (lh - center) / dist
            if np.any(rh):
                rh = (rh - center) / dist

    return np.concatenate([pose.flatten(), lh.flatten(), rh.flatten()])


def extract(results):
    pose = []
    if results.pose_landmarks:
        for idx in POSE_IDX:
            lm = results.pose_landmarks.landmark[idx]
            pose.extend([lm.x, lm.y, lm.z])
    else:
        pose = [0] * 15

    lh = []
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            lh.extend([lm.x, lm.y, lm.z])
    else:
        lh = [0] * 63

    rh = []
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            rh.extend([lm.x, lm.y, lm.z])
    else:
        rh = [0] * 63

    return normalize_frame(np.array(pose + lh + rh, dtype=np.float32))


def pad(seq):
    if len(seq) < MAX_SEQ_LENGTH:
        seq = np.vstack([seq, np.zeros((MAX_SEQ_LENGTH - len(seq), FEATURES))])
    return seq[:MAX_SEQ_LENGTH]


# ── Core processing ──────────────────────────────────────────────────────────
def process_video(video_path: Path, out_dir: Path, holistic):
    cap = cv2.VideoCapture(str(video_path))
    seq = []

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        res = holistic.process(rgb)
        seq.append(extract(res))

    cap.release()

    if not seq:
        print(f"  ⚠  No frames read from {video_path.name} — skipped")
        return

    seq      = pad(np.array(seq))
    out_path = out_dir / (video_path.stem + ".npy")
    np.save(out_path, seq)
    print(f"  ✓  {video_path.name}  →  {out_path}  shape={seq.shape}")


def collect_videos(root: Path):
    """
    Returns a list of (video_path, label) tuples.
    - If root contains sub-folders  → each sub-folder name is the label.
    - If root contains videos directly → label is the root folder name.
    """
    pairs = []
    subdirs = [d for d in root.iterdir() if d.is_dir()]

    if subdirs:
        for sub in sorted(subdirs):
            for v in sorted(sub.iterdir()):
                if v.suffix.lower() in VIDEO_EXT:
                    pairs.append((v, sub.name))
    else:
        for v in sorted(root.iterdir()):
            if v.suffix.lower() in VIDEO_EXT:
                pairs.append((v, root.name))

    return pairs


# ── Main ─────────────────────────────────────────────────────────────────────
def main():
    print("=" * 55)
    print("  Bridging Silence — Landmark Extractor → test/")
    print("=" * 55)
    print()

    # Ask the user to type the path
    raw = input("Enter path to video folder: ").strip().strip('"').strip("'")

    if not raw:
        print("ERROR: No path entered. Exiting.")
        sys.exit(1)

    input_path = Path(raw)
    if not input_path.exists():
        print(f"ERROR: Path does not exist:\n  {input_path}")
        sys.exit(1)

    videos = collect_videos(input_path)
    if not videos:
        print(f"ERROR: No videos (.mp4 .avi .mov .mkv .webm) found in:\n  {input_path}")
        sys.exit(1)

    print(f"\nFound {len(videos)} video(s).")
    print(f"Output  →  {OUTPUT_ROOT.resolve()}\n")

    with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holo:
        for video_path, label in videos:
            out_dir = OUTPUT_ROOT / label
            out_dir.mkdir(parents=True, exist_ok=True)
            process_video(video_path, out_dir, holo)

    print(f"\nDONE — results saved to '{OUTPUT_ROOT.resolve()}'")


if __name__ == "__main__":
    main()


  Bridging Silence — Landmark Extractor → test/


Found 2 video(s).
Output  →  D:\BRIDGING SILENCE DATA COLLECTION PIPELINE\test



d:\BRIDGING SILENCE DATA COLLECTION PIPELINE\.venv_extract\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


  ✓  test.mp4  →  test\TEST\test.npy  shape=(120, 141)
  ✓  testt.mp4  →  test\TEST\testt.npy  shape=(120, 141)

DONE — results saved to 'D:\BRIDGING SILENCE DATA COLLECTION PIPELINE\test'


In [3]:
%pip install protobuf==4.25.3

  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 4.25.3 which is incompatible.
